In [ ]:
# ========== 导入：网页抓取 + 本地 Ollama 摘要 ==========

# 导入标准库 os：本格未直接用到，保留以便扩展读环境变量
import os
# 导入 requests：用 HTTP GET 抓取目标网页 HTML
import requests
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可清理的 DOM 树
from bs4 import BeautifulSoup
# 从 IPython.display 导入 Markdown / display：在笔记本里渲染模型返回的 Markdown
from IPython.display import Markdown, display
# 导入 ollama：Python 客户端，调用本机已拉取的本地大模型
import ollama

# ========== 常量：模型名与请求头 ==========

# 本地模型 id：需事先 `ollama pull llama3.2`；字符串必须与本机模型名一致
MODEL = "llama3.2"

# 可选请求头，降低被拦截概率（伪装成常见浏览器 User-Agent）
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}


# ========== Website：抓取并清洗网页正文 ==========

class Website:
    def __init__(self, url):
        """
        用 BeautifulSoup 根据给定 URL 创建 Website 对象
        """
        # 保存原始 URL，便于调试或后续扩展
        self.url = url
        # GET 网页；带上 HEADERS，减少部分站点直接拒绝脚本请求
        response = requests.get(url, headers=HEADERS)
        # 用 html.parser 解析响应字节为 BeautifulSoup 文档
        soup = BeautifulSoup(response.content, 'html.parser')
        # 取 <title> 文本；没有标题则给占位字符串（进 prompt，保持英文）
        self.title = soup.title.string if soup.title else "No title found"
        # 有 <body> 才清洗正文；否则正文置空
        if soup.body:
            # 删掉脚本/样式/图片/输入框等对摘要无用的节点
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            # 抽取纯文本：换行分隔、去掉首尾空白
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            # 无 body：正文为空串，后续摘要会很短或无实质内容
            self.text = ""


# ========== Prompt：system 定角色，user 塞网页内容 ==========

# system prompt：影响模型行为，保持英文原文（可运行 / 影响回答的字符串不翻译）
system_prompt = """You are an assistant that analyzes the contents of a website 
and provides a short summary, ignoring navigation text. Respond in markdown."""


def user_prompt_for(website):
    """根据 Website 对象拼出 user 消息：标题 + 正文，要求 Markdown 短摘要。"""
    # f-string 把标题与清洗后正文嵌进 prompt；正文很长时会占满上下文
    return f"""You are looking at a website titled {website.title}.
The contents of this website are as follows. Please provide a short summary in markdown. 
If it includes news or announcements, summarize these too.

{website.text}
"""


# ========== 调用链：抓取 → ollama.chat → 展示 ==========

def summarize(url):
    """对给定 URL：抓网页 → 组 messages → 调本地 Ollama → 返回摘要文本。"""
    # 实例化 Website：内部完成 GET + BeautifulSoup 清洗
    website = Website(url)
    # ollama.chat：本地 Chat Completions 风格接口；messages 含 system + user
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt_for(website)}
        ]
    )
    # Ollama 返回 dict；正文在 message.content
    return response['message']['content']


def display_summary(url):
    """生成摘要并以 Markdown 在笔记本中渲染。"""
    # 先拿到纯文本摘要
    summary = summarize(url)
    # display(Markdown(...))：把 Markdown 渲染成富文本，而不是纯 print
    display(Markdown(summary))


# ========== 使用示例：换 URL 即可测其他站点 ==========

# 使用示例：对 Edward Donner 个人站做本地 Llama 摘要（需 Ollama 已运行）
display_summary("https://edwarddonner.com")
